# 📓 Course Module 1: Contrastive Learning (SimCLR)
Welcome to the first practical module of our Self-Supervised Learning course. In this notebook, we will build **SimCLR** (A Simple Framework for Contrastive Learning of Visual Representations) from scratch using PyTorch and train/evaluate it on the **CIFAR-10 dataset**.

### The Intuition
The goal of contrastive learning is to learn representations by bringing **positive pairs** (similar items) closer together in a latent space, while pushing **negative pairs** (dissimilar items) apart.

In SimCLR, a positive pair is created by taking a single image and applying two different random augmentations to it (e.g., cropping, color jitter). Negative pairs are formed by all other images in the current batch.

![SimCLR Architecture](https://miro.medium.com/max/1400/1*OkaH-u-n3Cq35fB7fD1Ppw.png)
*(Image representation of SimCLR: An image $x$ is augmented into two views $x_i$ and $x_j$, passed through an encoder $f(\cdot)$ to get representations $h_i$ and $h_j$, and then projected via a multi-layer perceptron $g(\cdot)$ to get $z_i$ and $z_j$, where the contrastive loss is computed.)*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. The Data Augmentation Pipeline
In contrastive learning, **augmentations are the curriculum**. The model learns by identifying that two aggressively augmented versions of the *same* image (positive pairs) share the same underlying semantic identity.

If the augmentations are too weak, the task is too easy. If they are too strong, the semantic meaning is destroyed.

In [ ]:
class ContrastiveTransformations:
    """
    Generates two differently augmented views of the same image.
    SimCLR relies heavily on Random Crop and Color Jitter.
    """
    def __init__(self, base_transforms):
        self.base_transforms = base_transforms

    def __call__(self, x):
        # Generate two different views of the image
        view1 = self.base_transforms(x)
        view2 = self.base_transforms(x)
        return view1, view2

# Standard SimCLR augmentation pipeline for small images (e.g., CIFAR-10)
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

### 2. The SimCLR Architecture
The architecture consists of two main parts:
1. **$f(\cdot)$ - The Base Encoder:** Usually a ResNet. It extracts the representation vector $h$. After training, this is the part we keep for downstream tasks (like classification).
2. **$g(\cdot)$ - The Projection Head:** A 2-layer Multi-Layer Perceptron (MLP). It maps $h$ into a lower-dimensional space $z$ where the contrastive loss is applied. We discard this after training because it learns to be *too* invariant to augmentations.

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, projection_dim=128):
        super(SimCLR, self).__init__()
        
        # 1. Base Encoder: Load ResNet and adapt initial conv for 32x32 images
        resnet = models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.feature_dim = resnet.fc.in_features
        resnet.fc = nn.Identity()
        self.encoder = resnet
        
        # 2. Projection Head: 2-layer MLP (Linear -> ReLU -> Linear)
        self.projector = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.ReLU(),
            nn.Linear(self.feature_dim, projection_dim)
        )

    def forward(self, x):
        # Extract representation 'h'
        h = self.encoder(x)
        # Extract projection 'z'
        z = self.projector(h)
        return h, z

simclr_model = SimCLR().to(device)
print(simclr_model)

### 3. The InfoNCE Loss (NT-Xent)
The Normalized Temperature-scaled Cross Entropy (NT-Xent) loss is the heart of SimCLR.

Given a batch of $N$ images, we generate $2N$ augmented views. For a specific view, there is **1 positive partner** (the other view of the same image) and **$2N-2$ negative partners** (all other images in the batch).

We compute the cosine similarity between all pairs, scale by a `temperature` parameter $\tau$, and apply Cross-Entropy:

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp\left(\text{sim}(z_i, z_j)/\tau\right)}{\sum_k \exp\left(\text{sim}(z_i, z_k)/\tau\right)}$$

In [ ]:
class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(InfoNCELoss, self).__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        """
        z1: Projections of view 1 [Batch_size, Dim]
        z2: Projections of view 2 [Batch_size, Dim]
        """
        batch_size = z1.size(0)
        
        # Normalize the projections (required for cosine similarity)
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        
        # Concatenate all views: [2 * Batch_size, Dim]
        # Order: [z1_1, z1_2... z1_N, z2_1, z2_2... z2_N]
        z = torch.cat([z1, z2], dim=0)
        
        # Compute cosine similarity matrix: [2N, 2N]
        sim_matrix = torch.matmul(z, z.T) / self.temperature
        
        # Remove the diagonal (similarity of an image with itself = 1.0)
        # We replace the diagonal with a very large negative number so it's ignored by softmax
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)
        
        # Create the labels.
        # For z1_i, the positive pair is z2_i (which is at index i + batch_size)
        # For z2_i, the positive pair is z1_i (which is at index i)
        labels = torch.arange(batch_size).to(z.device)
        labels = torch.cat([labels + batch_size, labels], dim=0)
        
        # Apply standard Cross Entropy
        loss = F.cross_entropy(sim_matrix, labels)
        
        return loss

criterion = InfoNCELoss(temperature=0.5)

---
## 🧪 4. Real Dataset Pre-training & Evaluation (CIFAR-10)

Now we bring theory to practice with a real experiment on CIFAR-10!
We split our dataset into two phases:
1. **Unlabeled Pre-training (100% Data):** Pre-train `SimCLR` using only unlabeled augmented image pairs.
2. **Linear Probing Evaluation (10% Labeled Data):** Freeze `simclr_model.encoder`, train a linear classifier (`nn.Linear(512, 10)`) on a limited 10% subset (5,000 images), and measure downstream accuracy on the test set.

In [ ]:
# --- Datasets and DataLoaders Setup ---
unlabeled_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=ContrastiveTransformations(simclr_transform))
unlabeled_loader = DataLoader(unlabeled_trainset, batch_size=128, shuffle=True, num_workers=2)

standard_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

labeled_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=standard_transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

# Sample 10% subset (5,000 images)
np.random.seed(42)
indices = np.random.choice(len(labeled_trainset), size=5000, replace=False)
labeled_subset = Subset(labeled_trainset, indices)

labeled_loader = DataLoader(labeled_subset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

In [ ]:
# --- Step 1: Pre-training SimCLR on 100% Unlabeled CIFAR-10 ---
optimizer_simclr = torch.optim.Adam(simclr_model.parameters(), lr=1e-3)

print("Pre-training SimCLR model on 100% UNLABELED CIFAR-10 (5 epochs demo)...")
simclr_model.train()
for epoch in range(5):
    running_loss = 0.0
    for (x1, x2), _ in unlabeled_loader:
        x1, x2 = x1.to(device), x2.to(device)
        
        _, z1 = simclr_model(x1)
        _, z2 = simclr_model(x2)
        
        loss = criterion(z1, z2)
        
        optimizer_simclr.zero_grad()
        loss.backward()
        optimizer_simclr.step()
        running_loss += loss.item()
        
    print(f"  Epoch {epoch+1}/5 - SimCLR Loss: {running_loss / len(unlabeled_loader):.4f}")

In [ ]:
# --- Step 2: Linear Probing Evaluation on 10% Labeled Data ---
encoder = simclr_model.encoder
for param in encoder.parameters():
    param.requires_grad = False  # FREEZE ENCODER BACKBONE

classifier = nn.Linear(512, 10).to(device)
optimizer_linear = torch.optim.Adam(classifier.parameters(), lr=1e-2)
ce_loss_fn = nn.CrossEntropyLoss()

print("Training Linear Classifier on 10% Labeled CIFAR-10 (Frozen Encoder)...")
encoder.eval()
classifier.train()
for epoch in range(10):
    for inputs, targets in labeled_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        with torch.no_grad():
            features = encoder(inputs)
            
        outputs = classifier(features)
        loss = ce_loss_fn(outputs, targets)
        
        optimizer_linear.zero_grad()
        loss.backward()
        optimizer_linear.step()

# Evaluate downstream accuracy
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100.0 * correct / total

simclr_eval_model = nn.Sequential(encoder, classifier)
simclr_acc = evaluate(simclr_eval_model, test_loader)
print(f"➡️ SimCLR + Linear Probe Test Accuracy (10% labels): {simclr_acc:.2f}%")